# BCSS-WSSS Stage-2 v2 Breakthrough Pipeline (Gate-5 4-Arm Suite)
Thin Colab Pro+ (G4 / A100 / L4) runner calling `src/stage2_v2/`:
- `arm1_deeplab_cleankd`: DeepLabV3+ R101 + Clean bn7 Soft-KD + Dice + MIL + PolySGD + EMA
- `arm2_segformer_oeem`: SegFormer-B3 Dual-Head + KL-Soft OEEM + Dice + MIL + AdamW + EMA
- `arm3_convnext_selftrain`: ConvNeXt-Base FPN + Soft-SCE + Round-2 EMA Self-Training
- `arm4_dinov2_adapter`: DINOv2-ViT-B/14 Spatial-Prior Adapter + KL-Soft OEEM + EMA


In [ ]:
import importlib.metadata
import os
import random
import shutil
import subprocess
import sys
from pathlib import Path

def _ensure_transformers(min_ver=(5, 0, 0)):
    need_install = False
    try:
        ver_str = importlib.metadata.version("transformers").split("+")[0].split("a")[0].split("b")[0].split("rc")[0]
        parts = tuple(int(x) for x in ver_str.split(".")[:3])
        if parts < min_ver:
            need_install = True
    except Exception:
        need_install = True
    if need_install:
        print("[SETUP] Installing transformers>=5.0.0...", flush=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=5.0.0"], check=True)

_ensure_transformers()

import numpy as np
import torch
import torchvision
import transformers

IN_COLAB = "google.colab" in sys.modules
CHECKPOINT_DIR = Path("/content/checkpoints") if IN_COLAB else Path("./checkpoints")
ENSEMBLE_DIR = Path(os.environ.get("STAGE2_ENSEMBLE_DIR", str(CHECKPOINT_DIR / "ensemble")))
OUTPUT_DIR_S2 = Path("/content/stage2_results") if IN_COLAB else Path("./stage2_results")
MEMMAP_DIR = Path("/content/train_PM_soft") if IN_COLAB else Path("./train_PM_soft")
ZIP_EXPORT_S2 = Path("/content/SE374_Stage2_Training_Results.zip") if IN_COLAB else Path("./SE374_Stage2_Training_Results.zip")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
ENSEMBLE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR_S2.mkdir(parents=True, exist_ok=True)
MEMMAP_DIR.mkdir(parents=True, exist_ok=True)

CANDIDATES = [
    Path("/content/medical_wsss_repo"),
    Path("/content/medical-wsss"),
    Path.cwd(),
    Path.cwd().parent,
]
CODE_ROOT = next((str(p.resolve()) for p in CANDIDATES if (p / "src" / "models").is_dir()), None)
if CODE_ROOT is None:
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/khoinguyen59/medical-wsss.git", "/content/medical_wsss_repo"], check=True)
    CODE_ROOT = str(Path("/content/medical_wsss_repo").resolve())

STAGE2_GIT_REF = os.environ.get("STAGE2_GIT_REF", "").strip()
if STAGE2_GIT_REF:
    if not (Path(CODE_ROOT) / ".git").exists():
        raise RuntimeError(f"STAGE2_GIT_REF={STAGE2_GIT_REF!r} requested, but {CODE_ROOT} is not a git repo")
    subprocess.run(["git", "-C", CODE_ROOT, "reset", "--hard", "HEAD"], capture_output=True, text=True, check=True)
    subprocess.run(["git", "-C", CODE_ROOT, "fetch", "origin", "+refs/heads/*:refs/remotes/origin/*", "--tags"], capture_output=True, text=True, check=False)
    subprocess.run(["git", "-C", CODE_ROOT, "fetch", "--depth", "1", "origin", STAGE2_GIT_REF], capture_output=True, text=True, check=False)
    _br = subprocess.run(["git", "-C", CODE_ROOT, "rev-parse", "--verify", "--quiet", f"refs/remotes/origin/{STAGE2_GIT_REF}^{{commit}}"], capture_output=True, text=True, check=False)
    if _br.returncode == 0:
        subprocess.run(["git", "-C", CODE_ROOT, "checkout", "-B", STAGE2_GIT_REF, f"refs/remotes/origin/{STAGE2_GIT_REF}"], capture_output=True, text=True, check=True)
        subprocess.run(["git", "-C", CODE_ROOT, "pull", "--ff-only", "origin", STAGE2_GIT_REF], capture_output=True, text=True, check=True)
    else:
        if (Path(CODE_ROOT) / ".git" / "shallow").exists():
            subprocess.run(["git", "-C", CODE_ROOT, "fetch", "--unshallow", "origin"], capture_output=True, text=True, check=False)
        subprocess.run(["git", "-C", CODE_ROOT, "checkout", "--force", STAGE2_GIT_REF], capture_output=True, text=True, check=True)
elif (Path(CODE_ROOT) / ".git").exists() and IN_COLAB:
    _hb = subprocess.run(["git", "-C", CODE_ROOT, "symbolic-ref", "--short", "-q", "HEAD"], capture_output=True, text=True, check=False)
    if _hb.returncode == 0 and _hb.stdout.strip():
        subprocess.run(["git", "-C", CODE_ROOT, "fetch", "origin", _hb.stdout.strip()], capture_output=True, text=True, check=False)
        subprocess.run(["git", "-C", CODE_ROOT, "pull", "--ff-only", "origin", _hb.stdout.strip()], capture_output=True, text=True, check=False)

for _k in [k for k in list(sys.modules) if k == "src" or k.startswith("src.")]:
    sys.modules.pop(_k, None)
if CODE_ROOT in sys.path:
    sys.path.remove(CODE_ROOT)
sys.path.insert(0, CODE_ROOT)

from src.stage2_v2.config import resolve_stage2_v2_config_from_env

cfg = resolve_stage2_v2_config_from_env()
STAGE2_MODE = os.environ.get("STAGE2_MODE", "train").strip().lower()
SEED = int(cfg.seed)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_DTYPE = (
    torch.bfloat16
    if (torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8 and torch.cuda.is_bf16_supported(including_emulation=False))
    else torch.float16
)
free_gb = shutil.disk_usage(MEMMAP_DIR).free / float(1024**3)
if not cfg.dry_run and STAGE2_MODE not in ("step0", "diag_r2_val") and free_gb < 18.0:
    raise RuntimeError(f"[STAGE2 ERROR] Disk headroom {free_gb:.2f} GB < 18.0 GB required.")

print(f"[READY] Code={CODE_ROOT} | Preset={cfg.preset_name} | Mode={STAGE2_MODE} | DryRun={cfg.dry_run}", flush=True)
print(f"[READY] Compute Device: {DEVICE} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})", flush=True)
print(f"[READY] Mixed Precision AMP Dtype: {AMP_DTYPE} | FreeDisk={free_gb:.2f}GB | torch={torch.__version__} | tv={torchvision.__version__} | tf={transformers.__version__}", flush=True)


## Step 2: Dataset Discovery & Strict Anti-Leakage Splits

In [ ]:
import urllib.request
import zipfile
from torch.utils.data import DataLoader
from src.colab_pkg.downloader import GDRIVE_DATASET_ID, HF_DATASET_URL, extract_patient_id, resolve_split_dirs
from src.training.dataset_loader import BCSSStage2ValDataset, load_bcss_patient_splits

DATASET_DIR = Path("/content/BCSS-WSSS") if IN_COLAB else Path("./data/BCSS-WSSS")
DATASET_ZIP = Path("/content/BCSS-WSSS.zip") if IN_COLAB else Path("./data/BCSS-WSSS.zip")
DATASET_DIR.parent.mkdir(parents=True, exist_ok=True)

def _resolve_bcss_train_dir(dataset_dir, min_images=10, min_patients=2, extra_roots=None):
    excluded_parts = {"sample_dataset", "repos", "drive_upload_ready", ".git"}
    if extra_roots is None:
        extra_roots = (
            Path("/content/BCSS-WSSS"),
            Path("/content"),
            Path("./data/BCSS-WSSS"),
            Path("./data"),
        )
    roots = [dataset_dir, dataset_dir / "BCSS-WSSS", *extra_roots]
    seen = set()
    candidates = []
    for root in roots:
        for rel in ("training", "training/img", "train", "train/img"):
            cand = root / rel
            key = str(cand.resolve()) if cand.exists() else str(cand)
            if key in seen:
                continue
            seen.add(key)
            if any(part.lower() in excluded_parts for part in cand.parts):
                continue
            if cand.is_dir():
                candidates.append(cand)
    for cand in candidates:
        pngs = sorted(cand.glob("*.png"))
        if len(pngs) < int(min_images):
            continue
        pids = {extract_patient_id(p.name) for p in pngs}
        if len(pids) >= int(min_patients):
            return cand
    return None

TRAIN_DIR = _resolve_bcss_train_dir(DATASET_DIR, min_images=10, min_patients=2)
if TRAIN_DIR is None:
    if not DATASET_ZIP.is_file() or DATASET_ZIP.stat().st_size < 1_000_000:
        print(f"[DOWNLOAD] Fetching BCSS-WSSS from {HF_DATASET_URL}", flush=True)
        try:
            urllib.request.urlretrieve(HF_DATASET_URL, str(DATASET_ZIP))
        except Exception:
            subprocess.run(["pip", "install", "-q", "gdown"], check=False)
            subprocess.run(["gdown", "--id", GDRIVE_DATASET_ID, "-O", str(DATASET_ZIP)], check=True)
    with zipfile.ZipFile(DATASET_ZIP, "r") as zf:
        zf.extractall(str(DATASET_DIR))
    TRAIN_DIR = _resolve_bcss_train_dir(DATASET_DIR, min_images=10, min_patients=2)

if TRAIN_DIR is None or not TRAIN_DIR.is_dir() or len(list(TRAIN_DIR.glob("*.png"))) == 0:
    raise FileNotFoundError(
        f"[DATASET ERROR] Missing or empty BCSS training directory (TRAIN_DIR={TRAIN_DIR}, DATASET_DIR={DATASET_DIR})."
    )

_, VAL_IMG_DIR, VAL_MASK_DIR = resolve_split_dirs(DATASET_DIR, split="val", min_images=1, min_patients=1)
if VAL_IMG_DIR is None:
    _, VAL_IMG_DIR, VAL_MASK_DIR = resolve_split_dirs(DATASET_DIR.parent, split="val", min_images=1, min_patients=1)
_, TEST_IMG_DIR, TEST_MASK_DIR = resolve_split_dirs(DATASET_DIR, split="test", min_images=1, min_patients=1)
if TEST_IMG_DIR is None:
    _, TEST_IMG_DIR, TEST_MASK_DIR = resolve_split_dirs(DATASET_DIR.parent, split="test", min_images=1, min_patients=1)
if VAL_IMG_DIR is None or VAL_MASK_DIR is None or not VAL_IMG_DIR.is_dir():
    raise FileNotFoundError(f"[DATASET ERROR] Missing val/img or val/mask under {DATASET_DIR}")
if TEST_IMG_DIR is None or TEST_MASK_DIR is None or not TEST_IMG_DIR.is_dir():
    raise FileNotFoundError(f"[DATASET ERROR] Missing test/img or test/mask under {DATASET_DIR}")

splits = load_bcss_patient_splits()
train_pids = splits["train"]
val_pids = splits["val"]
test_pids = splits["test"]
assert set(train_pids).isdisjoint(set(val_pids)) and set(test_pids).isdisjoint(set(train_pids) | set(val_pids)), "Patient leakage detected!"

sample_lim = 64 if cfg.dry_run else None
val_ds = BCSSStage2ValDataset(VAL_IMG_DIR, VAL_MASK_DIR, patient_ids=val_pids, limit=sample_lim, strict_patient_filter=True, seed=SEED)
test_ds = BCSSStage2ValDataset(TEST_IMG_DIR, TEST_MASK_DIR, patient_ids=test_pids, limit=sample_lim, strict_patient_filter=True, seed=SEED)
nw_eval = min(int(cfg.num_workers), os.cpu_count() or 2)
dl_kw = {"persistent_workers": True, "prefetch_factor": 2, "multiprocessing_context": "spawn"} if nw_eval > 0 else {}
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=nw_eval, pin_memory=(DEVICE.type == "cuda"), **dl_kw)
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False, num_workers=nw_eval, pin_memory=(DEVICE.type == "cuda"), **dl_kw)
print(f"[DATASET] Train={TRAIN_DIR} | Val={len(val_ds)} | Test={len(test_ds)}", flush=True)


In [ ]:
from src.colab_pkg.downloader import HF_STAGE1_URL, HF_STAGE2_URL
from src.models.wsss_inference_methods import WSSSTissueStage1NetCAM
from src.training.eval_stage1_full import load_stage1_strict

STAGE1_CKPT_SOURCE = os.environ.get("STAGE1_CKPT_SOURCE", "retrained").strip().lower()
STAGE1_BEST_CKPT = CHECKPOINT_DIR / "stage1_retrained_best.pth"
STAGE1_AUTHOR_CKPT = CHECKPOINT_DIR / "stage1_checkpoint_trained_on_bcss.pth"
if STAGE1_CKPT_SOURCE == "retrained":
    s1_ckpt_path = STAGE1_BEST_CKPT
    if not s1_ckpt_path.is_file():
        raise FileNotFoundError(f"[STAGE2 ERROR] Missing retrained Stage-1 ckpt: {s1_ckpt_path}")
else:
    s1_ckpt_path = STAGE1_AUTHOR_CKPT
    if not s1_ckpt_path.is_file():
        urllib.request.urlretrieve(HF_STAGE1_URL, str(s1_ckpt_path))

model_stage1 = WSSSTissueStage1NetCAM(n_class=4)
STAGE1_NORM_MODE, STAGE1_SHA256, s1_meta = load_stage1_strict(model_stage1, s1_ckpt_path)
model_stage1 = model_stage1.to(DEVICE).eval()
print(f"[STAGE1] Loaded {s1_ckpt_path.name} | sha256={STAGE1_SHA256[:12]} | norm={STAGE1_NORM_MODE}", flush=True)

STAGE1_CLS_CKPT = Path(os.environ.get("STAGE1_CLS_CKPT", str(CHECKPOINT_DIR / "stage1_best_cls.pth")))
model_stage1_cls = None
if STAGE1_CLS_CKPT.is_file():
    model_stage1_cls = WSSSTissueStage1NetCAM(n_class=4)
    _, cls_sha, _ = load_stage1_strict(model_stage1_cls, STAGE1_CLS_CKPT)
    model_stage1_cls = model_stage1_cls.to(DEVICE).eval()
    print(f"[STAGE1-CLS] Loaded Dual-Expert {STAGE1_CLS_CKPT.name} | sha256={cls_sha[:12]}", flush=True)

AUTHOR_STAGE2_CKPT = CHECKPOINT_DIR / "stage2_checkpoint_trained_on_bcss.pth"
if STAGE2_MODE == "step0" and not AUTHOR_STAGE2_CKPT.is_file():
    urllib.request.urlretrieve(HF_STAGE2_URL, str(AUTHOR_STAGE2_CKPT))


In [ ]:
import json
from src.models.wsss_inference_methods import safe_torch_load
from src.stage2_v2.ensemble import DEFAULT_ARM_NAMES
from src.stage2_v2.evaluator import (
    evaluate_r2_pseudo_mask_val_diagnostic,
    run_step0_calibration_and_author_anchor,
)
from src.stage2_v2.memmap_dataset import (
    generate_bn7_soft_memmap,
    generate_cross_family_r2_memmap,
)

def _resolve_r2_teachers(ensemble_dir):
    raw = os.environ.get("STAGE2_R2_TEACHER_ARMS", "").strip()
    names = [a.strip().lower() for a in raw.split(",") if a.strip()] if raw else list(DEFAULT_ARM_NAMES)
    found = {}
    for arm_name in names:
        short = arm_name.split("_")[0]
        for d in (ensemble_dir, CHECKPOINT_DIR):
            for cand in (d / f"{arm_name}.pth", d / f"{short}.pth", d / f"{short}_best.pth"):
                if cand.is_file():
                    found[arm_name] = cand
                    break
            if arm_name in found:
                break
    if len(found) < len(names):
        for pth in sorted(ensemble_dir.glob("*.pth")):
            try:
                meta = safe_torch_load(pth, device=torch.device("cpu"))
                preset = str(meta.get("preset", "")).strip().lower() if isinstance(meta, dict) else ""
                if preset in names and preset not in found:
                    found[preset] = pth
            except Exception:
                pass
    missing = [n for n in names if n not in found]
    if missing:
        raise FileNotFoundError(f"[R2 ERROR] Missing teacher checkpoints for {missing} in {ensemble_dir}")
    return [found[n] for n in names]

if STAGE2_MODE == "step0":
    step0_report = run_step0_calibration_and_author_anchor(
        stage1_model=model_stage1,
        val_loader=val_loader,
        test_loader=test_loader,
        author_stage2_ckpt=AUTHOR_STAGE2_CKPT,
        stage1_norm_mode=STAGE1_NORM_MODE,
        stage1_sha256=STAGE1_SHA256,
        device=DEVICE,
        output_json_path=OUTPUT_DIR_S2 / "STAGE2_TRAINING_METRICS.json",
    )
    print("[STEP0] Complete. Summary:", step0_report["bn7_soft_recipe_sweep_val"], flush=True)
elif STAGE2_MODE == "diag_r2_val":
    teacher_paths = _resolve_r2_teachers(ENSEMBLE_DIR)
    diag_soft_mode = os.environ.get("STAGE2_SOFT_MODE", "").strip().lower() or "sg_dap_soft"
    if diag_soft_mode == "ens4_r2_soft":
        diag_soft_mode = "sg_dap_soft"
    diag_report = evaluate_r2_pseudo_mask_val_diagnostic(
        val_loader=val_loader,
        teacher_ckpt_paths=teacher_paths,
        stage1_model=model_stage1,
        stage1_norm_mode=STAGE1_NORM_MODE,
        stage1_sha256=STAGE1_SHA256,
        soft_mode=diag_soft_mode,
        soft_temp=cfg.soft_temp,
        blend=cfg.r2_blend,
        r2_temp=1.0,
        track=cfg.track,
        device=DEVICE,
        amp_dtype=AMP_DTYPE,
        output_json_path=OUTPUT_DIR_S2 / "STAGE2_TRAINING_METRICS.json",
    )
    _diag_json = OUTPUT_DIR_S2 / "STAGE2_R2_VAL_DIAGNOSTIC.json"
    _diag_tmp = _diag_json.with_suffix(".json.tmp")
    _diag_tmp.write_text(json.dumps(diag_report, indent=2), encoding="utf-8")
    os.replace(_diag_tmp, _diag_json)
    print("[DIAG_R2_VAL] Complete:", diag_report["candidates"], "| go_decision=", diag_report["go_decision"], flush=True)
else:
    S1_MEMMAP_PATH, S1_MANIFEST_PATH, mm_info = generate_bn7_soft_memmap(
        stage1_model=model_stage1,
        train_img_dir=TRAIN_DIR,
        output_dir=MEMMAP_DIR,
        patient_ids=train_pids,
        stage1_norm_mode=STAGE1_NORM_MODE,
        stage1_sha256=STAGE1_SHA256,
        soft_mode=cfg.soft_mode,
        soft_temp=cfg.soft_temp,
        batch_size=max(16, int(cfg.batch_size)),
        device=DEVICE,
        max_samples=64 if cfg.dry_run else None,
        strict_patient_filter=True,
    )
    memmap_src = str(getattr(cfg, "memmap_source", os.environ.get("STAGE2_MEMMAP_SOURCE", "stage1"))).strip().lower()
    if memmap_src == "ens4_r2":
        teacher_paths = _resolve_r2_teachers(ENSEMBLE_DIR)
        _track_norm = str(getattr(cfg, "track", "track_a_clean")).strip().lower()
        _is_track_a = _track_norm in ("track_a_clean", "track_a")
        _r2_dat_name = "pm_soft_ens4_r2.dat" if _is_track_a else "pm_soft_ens4_r2_trackb.dat"
        _r2_man_name = "pm_soft_ens4_r2_manifest.json" if _is_track_a else "pm_soft_ens4_r2_trackb_manifest.json"
        MEMMAP_PATH, MANIFEST_PATH, mm_info = generate_cross_family_r2_memmap(
            teacher_ckpt_paths=teacher_paths,
            train_img_dir=TRAIN_DIR,
            s1_memmap_path=S1_MEMMAP_PATH,
            s1_manifest_path=S1_MANIFEST_PATH,
            output_dir=MEMMAP_DIR,
            patient_ids=train_pids,
            blend=cfg.r2_blend,
            r2_temp=1.0,
            track=cfg.track,
            batch_size=max(16, int(cfg.batch_size)),
            device=DEVICE,
            amp_dtype=AMP_DTYPE,
            max_samples=64 if cfg.dry_run else None,
            strict_patient_filter=True,
            dat_filename=_r2_dat_name,
            manifest_filename=_r2_man_name,
        )
        assert abs(float(mm_info["blend"]) - float(cfg.r2_blend)) < 1e-6, f"[R2 ERROR] blend mismatch: {mm_info['blend']} vs {cfg.r2_blend}"
        print(f"[R2 TELEMETRY] blend={mm_info['blend']} | mc_ent_p_ens4={mm_info.get('mean_entropy_p_ens4_uncal')} | mc_ent_q_r2={mm_info.get('mean_entropy_q_r2')} | argmax_disagree={mm_info.get('argmax_disagreement_p_ens4_vs_s1')} | frac_psum_lt_0_01={mm_info.get('frac_pixels_psum_lt_0_01')}", flush=True)
    else:
        MEMMAP_PATH, MANIFEST_PATH = S1_MEMMAP_PATH, S1_MANIFEST_PATH
    print(f"[MEMMAP] Ready ({memmap_src}): {MEMMAP_PATH} ({mm_info['num_patches']} patches, single_short_circuit={mm_info['single_label_short_circuit']})", flush=True)


In [ ]:
from dataclasses import replace
from src.stage2_v2.models import build_stage2_v2_model
from src.stage2_v2.trainer import train_stage2_v2

if STAGE2_MODE not in ("step0", "diag_r2_val"):
    run_cfg = replace(cfg, epochs=1) if cfg.dry_run else cfg
    model_stage2 = build_stage2_v2_model(run_cfg, pretrained=True)
    STAGE2_LATEST_CKPT = CHECKPOINT_DIR / "stage2_latest.pth"
    metrics = train_stage2_v2(
        cfg=run_cfg,
        model=model_stage2,
        train_img_dir=TRAIN_DIR,
        memmap_path=MEMMAP_PATH,
        manifest_path=MANIFEST_PATH,
        val_loader=val_loader,
        test_loader=test_loader,
        stage1_model=model_stage1,
        stage1_cls_model=model_stage1_cls,
        stage1_norm_mode=STAGE1_NORM_MODE,
        stage1_sha256=STAGE1_SHA256,
        train_patient_ids=train_pids,
        checkpoint_dir=CHECKPOINT_DIR,
        output_dir=OUTPUT_DIR_S2,
        zip_path=ZIP_EXPORT_S2,
        device=DEVICE,
        amp_dtype=AMP_DTYPE,
        max_train_samples=64 if run_cfg.dry_run else None,
        strict_patient_filter=True,
        resume_from=STAGE2_LATEST_CKPT if STAGE2_LATEST_CKPT.is_file() else None,
    )
    print("[DONE] Best VAL mIoU:", metrics["best_val_miou"], "| Calibrated Test:", metrics.get("test_eval_calibrated"), flush=True)


In [ ]:
import os
import zipfile

if STAGE2_MODE in ("step0", "diag_r2_val"):
    _tmp_zip = ZIP_EXPORT_S2.with_suffix(".zip.tmp")
    with zipfile.ZipFile(_tmp_zip, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in sorted(OUTPUT_DIR_S2.glob("*")):
            if f.is_file():
                zf.write(f, arcname=f.name)
    os.replace(_tmp_zip, ZIP_EXPORT_S2)

if IN_COLAB and ZIP_EXPORT_S2.is_file():
    try:
        from google.colab import files
        files.download(str(ZIP_EXPORT_S2))
    except Exception as exc:
        print(f"[NOTICE] Colab browser download skipped: {exc}", flush=True)


### Output Artifacts
- `/content/checkpoints/stage2_retrained_best.pth`
- `/content/checkpoints/stage2_latest.pth`
- `/content/stage2_results/STAGE2_TRAINING_METRICS.json`
- `/content/stage2_results/stage2_training_loss.png`
- `/content/SE374_Stage2_Training_Results.zip`